In [1]:
# Setup steps
import sys
import boto3
import sagemaker
from datetime import datetime
from sagemaker.model_monitor import (
    DataCaptureConfig,
    DatasetFormat,
    MonitoringOutput,
    ModelQualityMonitor,
)

from sagemaker.processing import (
    ProcessingInput,
    ProcessingOutput,
)
from sagemaker.transformer import Transformer
from sagemaker.model import Model


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
sess = sagemaker.Session()
bucket = sess.default_bucket()
role = sagemaker.get_execution_role()
region = boto3.Session().region_name
account_id = boto3.client("sts").get_caller_identity().get("Account")
sm_client = boto3.client('sagemaker')
model_package_group_name = f"FERModelGroupName"

In [3]:
# s3://sagemaker-us-east-1-188579839977/Team-project-data/model.tar.gz

In [4]:
# s3://sagemaker-us-east-1-188579839977/Team-project-data/baseline.csv

In [5]:
import sagemaker.model_monitor
print(dir(sagemaker.model_monitor))

['BaseliningJob', 'BatchTransformInput', 'BiasAnalysisConfig', 'ConstraintViolations', 'Constraints', 'CronExpressionGenerator', 'DataCaptureConfig', 'DataQualityDistributionConstraints', 'DataQualityMonitoringConfig', 'DatasetFormat', 'DefaultModelMonitor', 'EndpointInput', 'ExplainabilityAnalysisConfig', 'ModelBiasMonitor', 'ModelExplainabilityMonitor', 'ModelMonitor', 'ModelQualityMonitor', 'MonitoringDatasetFormat', 'MonitoringExecution', 'MonitoringOutput', 'NetworkConfig', 'Statistics', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'absolute_import', 'clarify_model_monitoring', 'cron_expression_generator', 'data_capture_config', 'data_quality_monitoring_config', 'dataset_format', 'model_monitoring', 'monitoring_alert', 'monitoring_files']


In [6]:
timestamp = datetime.utcnow().strftime("%Y-%m-%d-%H%M")

In [7]:
# S3 Paths:

s3_capture_upload_path = f"s3://{bucket}/group-5/data-capture/"
batch_output_s3_path = f"s3://{bucket}/grou-5/batch-transform-output/"
monitoring_results_s3_path = f"s3://{bucket}/group-5/monitoring-results/"
baseline_dataset_s3_path = f"s3://{bucket}/group-5/baseline/baseline.csv"


In [8]:
# Defining the model for Deployment:

model_data = f's3://{bucket}/group-5/models/model.tar.gz'  # ✅ Load model from S3
image_uri = sagemaker.image_uris.retrieve(
    framework="tensorflow",  # ✅ Change to TensorFlow
    version="2.10.0",  # ✅ Match your TensorFlow training version
    region=region,
    image_scope="inference",
    instance_type="ml.m5.xlarge"
)

model = Model(
    image_uri = image_uri,
    model_data = model_data,
    sagemaker_session = sess,
    role = role,
)


In [ ]:
# Creating a Batch Transformer and Batch Transform job:

print("Starting Batch Transform Job...")

transformer = model.transformer(
    instance_count = 1,
    instance_type = "ml.m5.xlarge",
    output_path = batch_output_s3_path,
    assemble_with = "Line",
    accept = "application/json",
)

transformer.transform(
    data = f"s3://{bucket}/test/test/Training_1070239.jpg",
    content_type = "application/jpeg",
    split_type = "None",
    wait = True
)

print(f"Batch transform job completed. Results stored at: {batch_output_s3_path}")

Starting Batch Transform Job...


INFO:sagemaker:Creating transform job with name: tensorflow-inference-2025-02-26-02-53-32-112


...........................INFO:__main__:PYTHON SERVICE: False
INFO:__main__:starting services
Traceback (most recent call last):
  File "/sagemaker/serve.py", line 502, in <module>
    ServiceManager().start()
  File "/sagemaker/serve.py", line 482, in start
    self._create_tfs_config()
  File "/sagemaker/serve.py", line 153, in _create_tfs_config
    raise ValueError("no SavedModel bundles found!")
ValueError: no SavedModel bundles found!


In [ ]:
# Creating a data quality monitor

print("Starting Data Quality Monitoring Job...")

data_quality_monitor = ModelQualityMonitor(
    role = role,
    instance_count = 1,
    instance_type = "ml.m5.xlarge",
    volume_size_in_gb = 30,
    max_runtime_in_seconds = 3600,
    base_job_name = "batch-monitoring-job", 
    sagemaker_session = sess
)

In [ ]:
# Running baseline job: 

print("Running batch monitoring job...")
data_quality_monitor.suggest_baseline(
    dataset_format = DatasetFormat.csv(header=True), 
    baseline_inputs = [
        ProcessingInput(
            source = baseline_dataset_s3_path,
            destination ="/opt/ml/processing/input/data"
        )
    ],
    output = ProcessingOutput(
        source = "/opt/ml/processing/output",
        destination = monitoring_results_s3_path
    ),
    wait = True,
    logs = True
)

print(f"Monitoring job completed. Baseline outputs stored at: {monitoring_results_s3_path}")

print("Batch transform and data quality monitoring setup completed successfully!")